# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
from dotenv import load_dotenv
import os
load_dotenv()
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
)

it = iter(ds)
for _ in range(3):
    print(next(it))

from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
fact_files = [f for f in files if "fact_content_daily_performance" in f]
print(len(fact_files))
for f in fact_files[:15]:
    print(f)

from huggingface_hub import hf_hub_download
import pandas as pd

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
)
df_march = pd.read_parquet(local_path)
print(df_march.shape)
print(df_march["report_date"].min(), df_march["report_date"].max())


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_fe8e8155ce1d47a2', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 5, 'gsc_clicks': 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (`content_hash_id`), aggregated over March 2026
(`report_date` between 2026-03-01 and 2026-03-31), using rows from
`fact_content_daily_performance` where `gsc_data_available` is True.

March was chosen as a mid-panel month, keeping earlier months (Jan–Feb)
and later months (Apr–Jun) untouched as a sealed window for future
label validation.

In [16]:
# Quick check: does the March slice actually match what I claimed above?
print("Row count (daily, before aggregation):", len(df_march))
print("Date range:", df_march["report_date"].min(), "-", df_march["report_date"].max())
print("Distinct pages:", df_march["content_hash_id"].nunique())
print("Distinct clients:", df_march["client_hash_id"].nunique())

Row count (daily, before aggregation): 9841378
Date range: 2026-03-01 - 2026-03-31
Distinct pages: 331437
Distinct clients: 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**Feature** (knowable before the decision moment):
- `gsc_impressions` — how often the page was shown
- `gsc_clicks` — how often it was clicked
- `gsc_avg_position` — average search ranking position

**Label / proxy** (what I'm ranking by — never a feature):
- `ctr_gap` — derived: expected CTR (per position tier) minus actual CTR.
  Not an observed outcome, a rule I define myself.

**Context** (for grouping/filtering — never for the model to learn from):
- `content_hash_id`, `client_hash_id` — identifiers, joins/grouping only
- `report_date` — used to build the March window, not a feature
- `gsc_data_available` — filter flag

**Excluded** (with a reason each):
- `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` — post-click, on-site behavior;
  my lane's question is about pre-click visibility (impressions vs clicks),
  not what happens after someone lands on the page
- `gsc_sum_position` — raw sum, only useful as an intermediate to compute
  `gsc_avg_position`; not used directly

In [17]:
feature_cols = ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
label_proxy = "ctr_gap"  # derived, not a raw column
context_cols = ["content_hash_id", "client_hash_id", "report_date", "gsc_data_available"]
excluded_cols = [c for c in df_march.columns if c not in feature_cols + context_cols + ["gsc_sum_position"]]

print("Feature:", feature_cols)
print("Label/proxy:", label_proxy)
print("Context:", context_cols)
print("Excluded (", len(excluded_cols), "cols ):", excluded_cols)

Feature: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Label/proxy: ctr_gap
Context: ['content_hash_id', 'client_hash_id', 'report_date', 'gsc_data_available']
Excluded ( 22 cols ): ['client_has_gsc', 'client_has_ga4', 'ga4_data_available', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
grain_check = (
    df_march
    .groupby(["content_hash_id", "client_hash_id", "report_date"])
    .size()
    .reset_index(name="row_count")
)
violations = grain_check[grain_check["row_count"] > 1]
print("Groups with more than 1 row:", len(violations))

Groups with more than 1 row: 0


In [19]:
gsc_available = df_march[df_march["gsc_data_available"] == True]
print("Rows with GSC data available:", len(gsc_available))
print("Total March rows:", len(df_march))
print("Share available: {:.1%}".format(len(gsc_available) / len(df_march)))
print("Date span:", gsc_available["report_date"].min(), "-", gsc_available["report_date"].max())

Rows with GSC data available: 3611061
Total March rows: 9841378
Share available: 36.7%
Date span: 2026-03-01 - 2026-03-31


In [20]:
availability_by_client = (
    df_march
    .groupby("client_hash_id")["gsc_data_available"]
    .mean()  # fraction of days with data available, per client
    .sort_values()
)
print("Clients with lowest GSC availability:")
print(availability_by_client.head())
print("\nClients with 0% availability (entirely missing this month):", (availability_by_client == 0).sum())

Clients with lowest GSC availability:
client_hash_id
client_19b89ee4fe3db6da    0.0
client_2910fd937f0b4d9a    0.0
client_2c32078d69f2cbad    0.0
client_4a18d1793d92fb84    0.0
client_770e8e5faa9cddfe    0.0
Name: gsc_data_available, dtype: float64

Clients with 0% availability (entirely missing this month): 8


Grain holds: 0 groups with more than 1 row for
(content_hash_id, client_hash_id, report_date).

Only 36.7% of March's daily rows have gsc_data_available = True
(3,611,061 of 9,841,378). 8 of 55 clients have zero GSC-available
rows this month entirely — their history likely starts later or
stopped before March.

In [21]:
page_month = (
    df_march
    .groupby(["content_hash_id", "client_hash_id"])
    .agg(
        gsc_impressions_month=("gsc_impressions", "sum"),
        gsc_clicks_month=("gsc_clicks", "sum"),
        gsc_position_sum=("gsc_sum_position", "sum"),
    )
    .reset_index()
)

# only pages that actually had at least 1 impression this month
page_month = page_month[page_month["gsc_impressions_month"] > 0].copy()

page_month["gsc_avg_position"] = page_month["gsc_position_sum"] / page_month["gsc_impressions_month"]
page_month["ctr"] = page_month["gsc_clicks_month"] / page_month["gsc_impressions_month"] * 100

print(page_month.shape)
page_month.head()

(176738, 7)


,content_hash_id,client_hash_id,gsc_impressions_month,gsc_clicks_month,gsc_position_sum,gsc_avg_position,ctr
0,content_000005d4ced12088,client_9958f0a7ae1df715,86,0,6199,72.081395,0.000000
2,content_00007bd2985b77c3,client_73cda7b4e4f265ea,47,0,249,5.297872,0.000000
6,content_0000cd28fbda69f3,client_3ffa76342f366962,29,0,111,3.827586,0.000000
8,content_0000d495bfbfb4a8,client_2094c6eb080311d5,15,0,32,2.133333,0.000000
11,content_00014efc121d911d,client_08a6a72ff48e62c0,116,1,672,5.793103,0.862069


In [23]:
def position_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

page_month["position_tier"] = page_month["gsc_avg_position"].apply(position_tier)
page_month["position_tier"].value_counts()

position_tier
page_1      83288
page_3_5    32240
striking    29922
top_3       18860
deep        12428
Name: count, dtype: int64

In [24]:
expected_ctr_by_tier = page_month.groupby("position_tier")["ctr"].mean()
print(expected_ctr_by_tier)

page_month["expected_ctr"] = page_month["position_tier"].map(expected_ctr_by_tier)
page_month["ctr_gap"] = page_month["expected_ctr"] - page_month["ctr"]
# positive ctr_gap = underperforming for its tier = editor should look at it
page_month[["content_hash_id", "position_tier", "ctr", "expected_ctr", "ctr_gap"]].sort_values("ctr_gap", ascending=False).head()

position_tier
deep        0.084632
page_1      0.487282
page_3_5    0.237885
striking    0.328465
top_3       1.169594
Name: ctr, dtype: float64


,content_hash_id,position_tier,ctr,expected_ctr,ctr_gap
286955,content_ddc993413ac8433e,top_3,0.0,1.169594,1.169594
286964,content_ddcb853b1ecac4e3,top_3,0.0,1.169594,1.169594
286881,content_ddba10f77a7dc7d2,top_3,0.0,1.169594,1.169594
286892,content_ddbb90abd64ee953,top_3,0.0,1.169594,1.169594
24777,content_13287652b0f9aa95,top_3,0.0,1.169594,1.169594


### Five features

1. `gsc_impressions_month` — available when: month closes, GSC data syncs
   daily, no future data needed.
2. `gsc_clicks_month` — available when: same as above, measured alongside
   impressions.
3. `gsc_avg_position` — available when: Google's own ranking measurement,
   known as soon as the month's daily rows are synced.
4. `ctr` — available when: derived from clicks/impressions above, so
   available at the same moment.
5. `position_tier` — available when: derived purely from `gsc_avg_position`,
   a fixed rule applied at aggregation time, no external data needed.

In [25]:
# --- DELIBERATE LEAK: do not do this in real modeling ---
page_month["leaky_feature"] = page_month["ctr_gap"] * 1.0  # literally the label in disguise

from scipy.stats import spearmanr

leaky_score, _ = spearmanr(page_month["leaky_feature"], page_month["ctr_gap"])
print("Quick ranking score WITH leaky feature:", leaky_score)

Quick ranking score WITH leaky feature: 1.0


In [26]:
# --- REMOVE THE LEAK ---
page_month = page_month.drop(columns=["leaky_feature"])

# --- HONEST SCORE: rank using only raw, pre-computed signals, not ctr_gap itself ---
# e.g. can raw ctr alone (without the position-tier adjustment) approximate the same ranking?
honest_score, _ = spearmanr(page_month["ctr"], page_month["ctr_gap"])
print("Honest ranking score (ctr alone, no leak):", honest_score)

Honest ranking score (ctr alone, no leak): -0.5929523405320412


### The leakage trap

I deliberately added `leaky_feature = ctr_gap` as a "feature" and measured
a quick ranking score (Spearman correlation) against the label itself:
the score was 1.0 — perfect, because the feature *was* the label in disguise.

I removed `leaky_feature` and measured an honest score using `ctr` alone
(without the position-tier adjustment): -0.59. This is negative because
`ctr_gap = expected_ctr - ctr`, so raw `ctr` is mechanically anti-correlated
with the gap, not because it's a genuinely poor signal. It's a reminder
that even "honest" features can sit close to the label's own formula —
a model built on this pair would need real features (position_tier as a
category, raw impressions/clicks volume) rather than a single column that
shadows the label's arithmetic.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [27]:
unavailable = df_march[df_march["gsc_data_available"] == False]
print(unavailable[["gsc_impressions", "gsc_clicks", "gsc_sum_position"]].describe())

       gsc_impressions  gsc_clicks  gsc_sum_position
count        6230317.0   6230317.0         6230317.0
mean               0.0         0.0               0.0
std                0.0         0.0               0.0
min                0.0         0.0               0.0
25%                0.0         0.0               0.0
50%                0.0         0.0               0.0
75%                0.0         0.0               0.0
max                0.0         0.0               0.0




- 63.3% of March's daily rows have gsc_data_available = False; verified
  these are not missing/corrupt data but true zero-impression days
  (all of gsc_impressions, gsc_clicks, gsc_sum_position = 0, std = 0).
  Summing over the month is unaffected by including or excluding them,
  but "days with zero impressions" is itself a signal worth keeping
  (e.g. as a future feature: days_with_impressions), not discarding.
- 8 of 55 clients have zero GSC-available rows in March entirely —
  likely their GSC history starts after March or ended before it.
  These clients are silently absent from my page-month aggregation
  this month, not scored as "bad" — a limitation, not a finding.
- This contract covers only GSC signals; GA4/on-site behavior after
  the click is out of scope for this lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.